In [ ]:
!pip install ultralytics roboflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 134.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

BASE_FOLDER = "/content/drive/MyDrive/CarParts"
os.makedirs(BASE_FOLDER, exist_ok=True)

print(f"Folder creado: {BASE_FOLDER}")

Folder creado: /content/drive/MyDrive/CarParts


In [ ]:
from ultralytics import YOLO
from roboflow import Roboflow

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import torch

In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

GPU: NVIDIA L4
VRAM (GB): 23.65915136


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="AmDk7XRgr1O3ZhfkpFZx")
project = rf.workspace("landebeau7").project("car-damage-type-detection-end-game-ranzq")
version = project.version(2)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Car-Damage-Type-Detection-End-Game-2 in yolov11:: 100%|██████████| 37869/37869 [00:04<00:00, 7801.61it/s] 


In [ ]:
!ls {dataset.location}

data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [ ]:
import os
import numpy as np
from pathlib import Path

In [ ]:
def check_labels(label_dir):
    """Detecta polígonos degenerados o coordenadas fuera de rango."""
    bad = []
    for f in Path(label_dir).rglob("*.txt"):
        lines = f.read_text().strip().splitlines()
        for i, line in enumerate(lines):
            vals = list(map(float, line.split()))
            coords = vals[1:]  # class_id, x1, y1, x2, y2, ...
            if any(v < 0 or v > 1 for v in coords):
                bad.append(f"{f.name}:{i} — coords fuera de [0,1]")
            if len(coords) < 6:  # mínimo 3 puntos para un polígono
                bad.append(f"{f.name}:{i} — polígono con menos de 3 puntos")
    return bad

In [ ]:
bad_labels = check_labels("/content/car_damage_defects/train/labels")
if bad_labels:
    print(f" {len(bad_labels)} issues with labels:")
    for b in bad_labels[:20]: print(" ", b)
else:
    print("Clear  Dataset")

Clear  Dataset


In [ ]:
import torch
from ultralytics import YOLO

In [ ]:
model = YOLO("yolo11m-seg.pt")

In [ ]:
def clip_gradients(trainer):
    torch.nn.utils.clip_grad_norm_(
        trainer.model.parameters(),
        max_norm=10.0
    )

In [ ]:
model.add_callback("on_before_optimizer_step", clip_gradients)

In [ ]:
results = model.train(
    data="/content/car_damage_defects/data.yaml",
    epochs=5,
    imgsz=640,
    batch=16,
    device=0,
    amp=False,

    optimizer="SGD",
    lr0=0.01,
    warmup_epochs=2.0,

    project="/content/drive/MyDrive/CarParts",
    name="verify_run",
    exist_ok=True,
    save=True,
    verbose=True,
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/car_damage_defects/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=verify_run, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, 

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch


In [ ]:
model = YOLO("/content/drive/MyDrive/CarParts/verify_run/weights/best.pt")

In [ ]:
model.add_callback("on_before_optimizer_step", clip_gradients)

In [ ]:

BEST_CKPT = "/content/drive/MyDrive/CarParts/verify_run/weights/best.pt"
YAML_PATH  = "/content/car_damage_defects/data.yaml"
DRIVE_OUT  = "/content/drive/MyDrive/CarParts"


In [ ]:
def clip_gradients(trainer):
    torch.nn.utils.clip_grad_norm_(
        trainer.model.parameters(), max_norm=10.0
    )
model.add_callback("on_before_optimizer_step", clip_gradients)


In [ ]:

results = model.train(
    data="/content/car_damage_defects/data.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    device=0,
    amp=False,

    optimizer="SGD",
    lr0=0.001,

    lrf=0.01,
    momentum=0.937,

    box=7,
    cls=1.5,
    dfl=1.5,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    mosaic=1.0,
    copy_paste=0.15,

    patience=7,
    workers=8,
    project=BASE_FOLDER,
    name="fine_tuning",
    exist_ok=False,
    verbose=True,
    plots=True,
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/car_damage_defects/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/CarParts/verify_run/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fine_tuning, nbs=64, nms=False, opset=None, opt

In [ ]:
metrics = model.val()
print(metrics)

In [ ]:
metrics_test = model.val(split="test")
print(metrics_test)

In [ ]:
image_path = "/content/car_damage_defects/test/images"

In [ ]:
import os
test_images = os.listdir(image_path)

img = os.path.join(image_path, test_images[0])

results = model(img, conf=0.25, iou=0.5)

boxes = results[0].boxes
num_objects = len(boxes)

print(f"Objetos detectados: {num_objects}")

results[0].show()

In [ ]:
len(results[0].boxes)

1

In [ ]:
import os
import glob
import json
import torch

In [ ]:
from ultralytics import YOLO
from PIL import Image
from google.colab.patches import cv2_imshow

In [ ]:
model_path = f"{BASE_FOLDER}/fine_tuning/weights/best.pt"
model = YOLO(model_path)

In [ ]:
test_image_dir = '/content/drive/MyDrive/CarParts/car_defects_imgs/'
image_extensions = ['*.jpg', '*.jpeg', '*.png']

In [ ]:
image_files = []

for ext in image_extensions:
    image_files.extend(glob.glob(os.path.join(test_image_dir, ext)))

In [ ]:
all_results_data = []

In [ ]:
print(f"--- Starting Inference on {len(image_files)} images ---")
for img_path in image_files:
    # Run inference
    # conf=0.25 is standard, adjust if you want more/fewer detections
    results = model.predict(source=img_path, conf=0.25, iou=0.6, imgsz=768)

    result = results[0]
    num_objects = len(result.boxes)
    filename = os.path.basename(img_path)

    # Display Image with Bounding Boxes
    print(f"\nFile: {filename} | Detected Objects: {num_objects}")
    annotated_img = result.plot() # Returns a numpy array (BGR)

    # Convert BGR to RGB for correct display
    display_img = Image.fromarray(annotated_img[..., ::-1])
    display(display_img)

    # Prepare JSON data for this image
    image_json = {
        "filename": filename,
        "object_count": num_objects,
        "detections": []
    }

    # Extract box coordinates and confidence
    for box in result.boxes:
        image_json["detections"].append({
            "class_id": int(box.cls[0]),
            "label": "Object",
            "confidence": float(box.conf[0]),
            "bbox_xyxy": box.xyxy[0].tolist() # [xmin, ymin, xmax, ymax]
        })

    all_results_data.append(image_json)

# 4. Print and Save JSON Output
final_json = json.dumps(all_results_data, indent=4)
print("\n--- Final Inference JSON Report ---")
#print(final_json)

In [ ]:
print(f"--- Starting Export to ONNX (Opset 18) ---")
success = model.export(
    format='onnx',
    opset=18,
    simplify=True,
    imgsz=640,
)

--- Starting Export to ONNX (Opset 18) ---
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/drive/MyDrive/CarParts/fine_tuning/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 42, 8400), (1, 32, 160, 160)) (43.1 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 223ms
Prepared 4 packages in 1.16s
Installed 4 packages in 237ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime==1.26.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 2.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 18...
ONNX: slimming wi